In [ ]:
# =============================================================================
# CROSS-ENCODER — APPLY for v4 (run in the Step 2 v4 notebook that gathered all countries; CPU)
# Uses CE-1c's choice (ce_config_v4.json): logistic blend or LightGBM stacker (stacker_v4.txt),
# on the chosen band. Writes /kaggle/working/stage2_v4ce_output/ + validator.
# INPUTS  ce_config_v4.json [+ stacker_v4.txt] (CE-1c) and BOTH cross-encoder score files:
#         ce_test.parquet (v3 band) and ce_test_v4.parquet (new v4 pairs).
# =============================================================================
import json, shutil, subprocess, sys
import numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path

W, I = Path("/kaggle/working"), Path("/kaggle/input")
DATA_ROOT = Path("/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset")
V3OUT, CKPT, OUTD = W / "stage2_v4_output", W / "ckpt_test", W / "stage2_v4ce_output"

# ---- stacker features (IDENTICAL in CE-1b and apply-v2) -------------------------
STACK_FEATURES = ["z_v3", "z_ce", "ce_rank", "ce_gap", "n_band", "v3_rank", "v3_gap",
                  "ent_n_conf", "is_s3", "cand_blank"]


def _logit_(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def stack_features(kept, band_mask, ce, cand_blank):
    """kept: DataFrame of ALL kept pairs with columns ent, cand, prob (v3).
    band_mask: bool array over kept rows (rows re-scored by the cross-encoder).
    ce: cross-encoder probability for the band rows (same order).
    cand_blank: 1 if the candidate's address is blank, for the band rows.
    Returns the feature matrix for the band rows."""
    k = kept[["ent", "prob"]].copy()
    g = k.groupby("ent")["prob"]
    k["v3_rank"] = g.rank(ascending=False, method="first")
    k["v3_gap"] = g.transform("max") - k["prob"]
    k["ent_n_conf"] = (k["prob"] >= 0.5).groupby(k["ent"]).transform("sum")
    b = k[band_mask].copy()
    b["ce"] = ce
    gb = b.groupby("ent")["ce"]
    b["ce_rank"] = gb.rank(ascending=False, method="first")
    b["ce_gap"] = gb.transform("max") - b["ce"]
    b["n_band"] = gb.transform("size")
    b["z_v3"], b["z_ce"] = _logit_(b["prob"]), _logit_(b["ce"])
    b["is_s3"] = kept.loc[band_mask, "cand"].astype(str).str.startswith("S3-").to_numpy().astype(np.float32)
    b["cand_blank"] = np.asarray(cand_blank, dtype=np.float32)
    return b[STACK_FEATURES].to_numpy(dtype=np.float32)


STACK_PARAMS = {"objective": "binary", "num_leaves": 31, "learning_rate": 0.05, "min_data_in_leaf": 50,
                "feature_fraction": 0.9, "bagging_fraction": 0.8, "bagging_freq": 1, "seed": 42, "verbose": -1}
STACK_ROUNDS = 400


cfg_f = [f for f in list(I.rglob("ce_config_v4.json")) + list(W.rglob("ce_config_v4.json"))
         if not json.load(open(f)).get("smoke")]
ce_f = (list(I.rglob("ce_test.parquet")) + list(W.glob("ce_test.parquet"))
        + list(I.rglob("ce_test_v4.parquet")) + list(W.glob("ce_test_v4.parquet")))
assert cfg_f and ce_f, "attach ce_config_v4.json (+ stacker_v4.txt) and the cross-encoder score files"
print("score files:", [str(f) for f in ce_f])
CE = json.load(open(cfg_f[0]))
assert CE["stack_features"] == STACK_FEATURES, "stacker feature list differs: use matching CE-1c / apply-v4 files"
_stk = [f for f in [cfg_f[0].parent / "stacker_v4.txt"] + list(I.rglob("stacker_v4.txt")) + list(W.rglob("stacker_v4.txt"))
        if f.exists()]
assert _stk or CE["method"] != "stacker", "stacker_v4.txt not found: attach it"
STK = lgb.Booster(model_file=str(_stk[0])) if CE["method"] == "stacker" else None
print("stacker:", _stk[0] if _stk else None, "| config:", cfg_f[0])
lo, hi = CE["band"]
# Countries scored with pred_early_stop_margin=8: probabilities exact only inside 0.018-0.982,
# so their band is clamped (must match the export cell). [] if none.
MARGIN8_COUNTRIES = []          # v4 was scored with margin 20: no clamp
CLAMP = (0.0180, 0.9820)
print(f"held-out: v4 {CE['heldout_f05_base']:.4f} -> {CE['method']} on band {CE['band']}: {CE['heldout_f05']:.4f}   use_ce={CE['use_ce']}")
ces = (pd.concat([pd.read_parquet(f) for f in ce_f], ignore_index=True)
       .drop_duplicates(["s1", "cand"]).set_index(["s1", "cand"])["ce"])
addr = pd.concat([pd.read_csv(DATA_ROOT / "test" / f"test_source{i}.tsv", sep="\t", dtype=str,
                              keep_default_na=False, usecols=[0, 2]) for i in (2, 3)], ignore_index=True)
BLANK = dict(zip(addr.iloc[:, 0], addr.iloc[:, 1].str.strip() == ""))
del addr
OUTD.mkdir(exist_ok=True)
R, stats = {}, []
for part in sorted((V3OUT / "_parts").glob("*.npz")):
    c = part.stem
    z = np.load(part)
    nq = int(z["nq"][0])
    qids = np.load(CKPT / c / "qids.npy", allow_pickle=True)[:nq]
    pids = np.load(CKPT / c / "pids.npy", allow_pickle=True)
    q, p, prob = z["q"], z["p"], z["prob"].astype(np.float64)
    kept = pd.DataFrame({"ent": q, "cand": pids[p], "prob": prob})
    lo_c, hi_c = (max(lo, CLAMP[0]), min(hi, CLAMP[1])) if c in MARGIN8_COUNTRIES else (lo, hi)
    band = (prob >= lo_c) & (prob < hi_c)
    ce = ces.reindex(pd.MultiIndex.from_arrays([qids[q[band]], pids[p[band]]])).to_numpy()
    ok = ~np.isnan(ce)
    bi = np.flatnonzero(band)
    use = np.zeros(len(q), bool); use[bi[ok]] = True          # band rows that have a cross-encoder score
    score = prob.copy()
    if use.any():
        cb = np.array([BLANK.get(x, False) for x in kept.loc[use, "cand"]], dtype=np.float32)
        if STK is not None:
            score[use] = STK.predict(stack_features(kept, use, ce[ok], cb))
        else:
            z1, z2 = _logit_(prob[use]), _logit_(ce[ok])
            zz = np.column_stack([z1, z2, z1 * z2]) @ np.asarray(CE["blend_coef"]) + CE["blend_intercept"]
            score[use] = 1 / (1 + np.exp(-zz))
    t = CE["thresholds"].get(c, CE["threshold_unseen"])
    own = pd.Series(-score).groupby(p).rank(method="first").to_numpy() == 1   # exactly one owner
    sel = (score >= t) & own                                   # threshold + one-owner
    R[c] = dict(q=q, p=p, sel=sel, qids=qids, pids=pids, nq=nq)
    per = np.bincount(q[sel], minlength=nq)
    stats.append((c, t, int(band.sum()), int((~ok).sum()), per.mean(), (per == 0).mean()))

print(f"\n  {'country':<8}{'threshold':>10}{'band pairs':>12}{'no CE score':>12}{'matches/entity':>16}{'empty rows':>12}")
for c, t, nb, nm, mpe, er in stats:
    print(f"  {c:<8}{t:>10.2f}{nb:>12,}{nm:>12,}{mpe:>16.2f}{er*100:>11.1f}%")
if any(nm > 0 for *_, nm, _, _ in [(s[0], s[1], s[2], s[3], s[4], s[5]) for s in stats]):
    print("  NOTE: some band pairs have no cross-encoder score (band mismatch between export and CE-1b?): "
          "they keep their v3 score.")
with open(OUTD / "matching_results.tsv", "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for c in sorted(R):
        r = R[c]
        qa, pa = r["q"][r["sel"]], r["p"][r["sel"]]
        bnd = np.searchsorted(qa, np.arange(r["nq"] + 1))
        for i in range(r["nq"]):
            f.write(f"{r['qids'][i]}\t{','.join(r['pids'][pa[bnd[i]:bnd[i+1]]])}\n")
shutil.copyfile(V3OUT / "candidate_pairs.tsv", OUTD / "candidate_pairs.tsv")
r = subprocess.run([sys.executable, str(DATA_ROOT.parent / "utils" / "validate_submission.py"),
                    "--matching", str(OUTD / "matching_results.tsv"), "--candidate", str(OUTD / "candidate_pairs.tsv"),
                    "--test-dir", str(DATA_ROOT / "test"), "--check-ids"], capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-400:])
print(f"exit code {r.returncode}: {'PASS' if r.returncode == 0 else 'FAIL / killed — see above'}")
print(f"SUBMIT: {OUTD / 'matching_results.tsv'}")